# Financial Metrics - Latest WSB Stocks
This notebook refreshes financial metrics for the latest WallStreetBets tickers using current Yahoo Finance data.
It now:
- loads the latest top WSB stocks,
- fetches current Yahoo Finance price history,
- computes rolling standard deviation, SMA, EWMA, RSI, and MACD,
- runs a linear regression / RMSE check for each selected ticker,
- saves refreshed CSVs and charts under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    compute_price_features,
    evaluate_linear_regression,
    fetch_price_history,
    plot_symbol_dashboard,
    run_pipeline,
)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "1y"
TOP_K = 4
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None  # Example: ["AMD", "TSLA"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
if REFRESH_WSB_DATA or (not summary_path.exists()):
    pipeline_result = run_pipeline(top_n=10, per_feed=100, price_period=PRICE_PERIOD, output_dir=OUTPUT_DIR)
    print(json.dumps(pipeline_result, indent=2))
summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "latest_mention"]]

,ticker,mention_count,post_count,avg_sentiment,latest_mention
0,WEN,32,18,0.261522,2026-06-28 08:32:25+00:00
1,MSFT,9,7,0.165829,2026-06-28 19:43:16+00:00
2,NVDA,6,3,0.247767,2026-06-27 15:42:12+00:00
3,SLS,5,3,0.007333,2026-06-28 17:29:11+00:00
4,MSTR,5,2,-0.538050,2026-06-26 14:15:39+00:00
5,CEG,7,1,0.945400,2026-06-28 16:14:31+00:00
6,MU,3,2,0.148500,2026-06-24 21:09:17+00:00
7,SPCX,2,2,0.146200,2026-06-25 15:19:32+00:00
8,JPM,2,2,0.683050,2026-06-28 22:33:23+00:00
9,SPY,2,2,0.223100,2026-06-26 20:18:36+00:00


## Step 4 - Select tickers for financial analysis

In [4]:
if MANUAL_TICKERS:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()
print("Selected tickers:", selected_tickers)

Selected tickers: ['WEN', 'MSFT', 'NVDA', 'SLS']


## Step 5 - Fetch current price history and compute financial metrics

In [5]:
def safe_latest_value(in_df, column_name, caster=float):
    if in_df.empty or column_name not in in_df.columns:
        return None
    col_data = in_df[column_name].dropna()
    if col_data.empty:
        return None
    return caster(col_data.iloc[-1])


price_feature_frames = {}
regression_frames = {}
analysis_rows = []
for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    features_df = compute_price_features(history_df)
    rmse, regression_df = evaluate_linear_regression(features_df)
    features_path = PRICES_DIR / f"{ticker.lower()}_price_features.csv"
    regression_path = PRICES_DIR / f"{ticker.lower()}_regression_results.csv"
    dashboard_path = CHARTS_DIR / f"{ticker.lower()}_dashboard.png"
    features_df.to_csv(features_path)
    regression_df.to_csv(regression_path)
    plot_symbol_dashboard(ticker, features_df, regression_df, dashboard_path)
    price_feature_frames[ticker] = features_df
    regression_frames[ticker] = regression_df
analysis_rows.append({
        "ticker": ticker,
        "rows": int(len(features_df)),
        "last_close": safe_latest_value(features_df, "Close", float),
        "latest_rsi": safe_latest_value(features_df, "RSI", float),
        "latest_macd": safe_latest_value(features_df, "MACD", float),
        "return_21d": safe_latest_value(features_df, "21D Return", float),
        "rmse": rmse,
        "features_csv": str(features_path),
        "regression_csv": str(regression_path),
        "dashboard_png": str(dashboard_path),
    })
analysis_df = pd.DataFrame(analysis_rows).sort_values("ticker").reset_index(drop=True)
analysis_summary_path = OUTPUT_DIR / "financial_metrics_summary.csv"
analysis_df.to_csv(analysis_summary_path, index=False)
analysis_df

,ticker,rows,last_close,latest_rsi,latest_macd,return_21d,rmse,features_csv,regression_csv,dashboard_png
0,SLS,251,12.39,78.741904,0.845672,0.444056,0.422731,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...


## Step 6 - Inspect the computed metrics for each selected ticker

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} latest metrics =====")
    display_cols = [
        "Close",
        "Volume",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    print(price_feature_frames[ticker][display_cols].tail(5).to_string())


===== WEN latest metrics =====
            Close     Volume  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI      MACD  5D Return  21D Return
Date                                                                                                                                 
2026-06-22   6.17   18523600            0.564367            7.270000             7.286574  34.246931 -0.193994  -0.091311   -0.203871
2026-06-23   6.26    7573200            0.593643            7.235333             7.263050  36.426613 -0.222324  -0.094067   -0.178478
2026-06-24   7.86  210481900            0.597232            7.272000             7.276728  61.109017 -0.114351   0.161004    0.006402
2026-06-25   7.33  102681100            0.585516            7.253000             7.277948  53.674979 -0.070733   0.054676   -0.004076
2026-06-26   7.80   80255600            0.568064            7.240000             7.289908  58.496768  0.001739   0.147059    0.041389

===== MSFT latest metrics ===

## Step 7 - Plot the latest closing prices for the selected tickers

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in selected_tickers:
    close_series = price_feature_frames[ticker]["Close"].dropna()
    ax.plot(close_series.index, close_series.values, label=ticker, linewidth=1.8)
ax.set_title("Latest selected WSB tickers - close prices")
ax.set_ylabel("Close")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 - Review the regression predictions

In [8]:
for ticker in selected_tickers:
    regression_df = regression_frames[ticker]
    print(f"\n===== {ticker} regression results =====")
    if regression_df.empty:
        print("Not enough data for regression output.")
    else:
        print(regression_df.tail(10).to_string())


===== WEN regression results =====
            Close  Predicted Close
Date                              
2026-06-12   6.79         6.734702
2026-06-15   6.91         6.856544
2026-06-16   6.77         6.759769
2026-06-17   6.95         6.962248
2026-06-18   6.80         6.813489
2026-06-22   6.17         6.339083
2026-06-23   6.26         6.381375
2026-06-24   7.86         7.274807
2026-06-25   7.33         7.125187
2026-06-26   7.80         7.434285

===== MSFT regression results =====
                 Close  Predicted Close
Date                                   
2026-06-12  390.739990       387.446190
2026-06-15  399.760010       393.960555
2026-06-16  393.829987       389.460914
2026-06-17  378.910004       377.309176
2026-06-18  379.399994       374.884699
2026-06-22  367.339996       365.128736
2026-06-23  373.940002       366.334406
2026-06-24  365.459991       359.278491
2026-06-25  352.829987       349.229060
2026-06-26  372.970001       349.644416

===== NVDA regression resu

## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "price_period": PRICE_PERIOD,
    "top_k": TOP_K,
    "analysis_summary_csv": str(analysis_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "WEN",
    "MSFT",
    "NVDA",
    "SLS"
  ],
  "price_period": "1y",
  "top_k": 4,
  "analysis_summary_csv": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/financial_metrics_summary.csv",
  "output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
